# Clipt V5 — YouTube Auto-Label + Video Classifier + Jersey OCR (Fully Autonomous)

Run **Runtime → Run All** and walk away. Every model downloads automatically after training.

| Section | What it does | Models |
|---------|-------------|--------|
| **0** | Setup + helpers | — |
| **1** | YouTube Highlight Harvester (~30 vids/sport) | — |
| **2** | Claude Vision Auto-Labeling | — |
| **3** | VideoMAE — Basketball | `videomae_basketball_v5/` |
| **4** | VideoMAE — Football | `videomae_football_v5/` |
| **5** | VideoMAE — Lacrosse | `videomae_lacrosse_v5/` |
| **6** | YOLO Outcome Classifier | 3 `.pt` models |
| **7** | Jersey OCR v5 + Player Detector v5 | 2 `.pt` models |
| **8** | Final Summary + Diagnostics | — |

**Total: 8 models** (3 VideoMAE + 3 YOLO outcome + 1 jersey OCR + 1 player detector)
**Runtime: ~10 hours on A100**
**Colab Secrets: `ANTHROPIC_API_KEY` (required)**

### IF COLAB DISCONNECTS
1. Reconnect and run **Cell 0A** (install) and **Cell 0B** (imports)
2. Run **Cell 0C** (helpers) and **Cell 0D** (configs)
3. Skip to whichever section you were on — labels and datasets are saved to disk

### WHAT THESE MODELS DO
- **VideoMAE**: Classifies entire video clips into play types (made_shot, touchdown, etc.)
- **YOLO Outcome**: Frame-level play classification (same play types, per-frame)
- **Jersey OCR v5**: Finds and reads jersey numbers in video frames (the most critical model)
- **Player Detector v5**: Finds player bounding boxes to isolate individuals


---
## SECTION 0: Setup

Install dependencies, imports, GPU check, helper functions, sport configs.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0A — Install Dependencies
# ═══════════════════════════════════════════════════════
!pip install yt-dlp "scenedetect[opencv]" anthropic transformers pytorchvideo evaluate accelerate ultralytics supabase pyyaml opencv-python-headless matplotlib scikit-learn onnx onnxruntime -q
print("\n✅ All dependencies installed")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0B — Imports + GPU Check + Colab Secrets
# ═══════════════════════════════════════════════════════
from google.colab import userdata, files
import os, sys, torch, shutil, glob, json, time, traceback, gc
import random, re, base64, subprocess
from pathlib import Path
from collections import Counter, defaultdict
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── GPU check ──
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️  WARNING: No GPU detected. Training will be extremely slow.")

# ── Colab secrets ──
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
assert ANTHROPIC_API_KEY, "❌ Set ANTHROPIC_API_KEY in Colab Secrets (key icon in sidebar)"


try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = None

print(f"Anthropic API key: ✅ SET")
print(f"Roboflow: {'✅ SET' if ROBOFLOW_API_KEY else '⏭️  NOT SET (not needed for v5)'}")

TRAINING_LOG = []

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0C — Helper Functions
# ═══════════════════════════════════════════════════════
from anthropic import Anthropic
_claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)


def safe_download_video(url, output_dir, max_retries=3):
    """Download a YouTube video with yt-dlp, 720p max, mp4."""
    os.makedirs(output_dir, exist_ok=True)
    existing_before = set(os.listdir(output_dir))
    for attempt in range(max_retries):
        try:
            cmd = [
                "yt-dlp", "-f",
                "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best",
                "--merge-output-format", "mp4",
                "-o", os.path.join(output_dir, "%(id)s.%(ext)s"),
                "--no-overwrites", url
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                new_files = set(os.listdir(output_dir)) - existing_before
                for f in new_files:
                    if f.endswith('.mp4'):
                        full_path = os.path.join(output_dir, f)
                        if os.path.getsize(full_path) > 100_000:
                            return full_path
                # Already downloaded
                for f in sorted(os.listdir(output_dir)):
                    if f.endswith('.mp4'):
                        return os.path.join(output_dir, f)
            if attempt < max_retries - 1:
                print(f"    Retry {attempt+2}/{max_retries}...")
                time.sleep(2)
        except subprocess.TimeoutExpired:
            print(f"    Attempt {attempt+1} timed out")
        except Exception as e:
            print(f"    Attempt {attempt+1} error: {e}")
    return None


def safe_download_video_segment(url, output_path, start_time=None, end_time=None, max_retries=3):
    """Download a specific time segment from a YouTube video."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    for attempt in range(max_retries):
        try:
            cmd = [
                "yt-dlp", "-f",
                "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best",
                "--merge-output-format", "mp4",
                "-o", output_path, "--no-playlist",
            ]
            if start_time is not None and end_time is not None:
                # Use ffmpeg postprocessor for segment extraction
                cmd.extend([
                    "--download-sections", f"*{start_time}-{end_time}",
                    "--force-keyframes-at-cuts",
                ])
            cmd.append(url)
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode == 0 and os.path.exists(output_path):
                return output_path
            if attempt < max_retries - 1:
                time.sleep(2)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
    return None


def extract_frames_from_clip(clip_path, num_frames=4):
    """Extract evenly-spaced frames from a video clip as base64 JPEG."""
    cap = cv2.VideoCapture(clip_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []
    num_frames = min(num_frames, max(total_frames, 1))
    indices = [int(i * (total_frames - 1) / max(num_frames - 1, 1)) for i in range(num_frames)]
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            h, w = frame.shape[:2]
            if w > 512:
                scale = 512 / w
                frame = cv2.resize(frame, (512, int(h * scale)))
            _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
            frames.append(base64.b64encode(buf).decode('utf-8'))
    cap.release()
    return frames


def extract_frames_as_images(clip_path, num_frames=8):
    """Extract evenly-spaced frames from a video clip as numpy arrays (BGR)."""
    cap = cv2.VideoCapture(clip_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    return frames


def label_clip_with_claude(clip_path, sport, labels, max_retries=2):
    """Send 4 frames to Claude Sonnet for outcome labeling. Returns dict."""
    frames_b64 = extract_frames_from_clip(clip_path, num_frames=4)
    if not frames_b64:
        return {"label": "unknown", "confidence": 0.0, "description": "no frames extracted"}

    label_list = ", ".join(labels)
    content = []
    for i, b64 in enumerate(frames_b64):
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}})
        content.append({"type": "text", "text": f"Frame {i+1}/{len(frames_b64)}"})
    content.append({"type": "text", "text": (
        f"Analyze these sequential frames from a {sport} highlight clip.\n"
        f"Classify into ONE label: {label_list}\n"
        f"If unclear, use \"dead_ball\".\n"
        f"Respond with ONLY JSON: {{\"label\": \"x\", \"confidence\": 0.0-1.0, \"description\": \"brief\"}}"
    )})

    for attempt in range(max_retries):
        try:
            resp = _claude_client.messages.create(
                model="claude-sonnet-4-6", max_tokens=200,
                messages=[{"role": "user", "content": content}]
            )
            text = resp.content[0].text.strip()
            match = re.search(r'\{[^}]+\}', text)
            result = json.loads(match.group()) if match else {"label": "unknown", "confidence": 0.0, "description": text[:100]}
            if result.get("label") not in labels and result.get("label") not in ("dead_ball", "unknown"):
                result["label"] = "unknown"
            return result
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return {"label": "unknown", "confidence": 0.0, "description": f"error: {e}"}
    return {"label": "unknown", "confidence": 0.0, "description": "exhausted retries"}


def safe_train(model_name, data_path, epochs, imgsz=640, batch=8, task="classify", base_model=None, **kwargs):
    """YOLO training with OOM retry + crash recovery for autopilot."""
    from ultralytics import YOLO

    if data_path is None:
        print(f"⏭️  {model_name}: data_path is None, SKIPPING")
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, 'data_path is None'))
        return False

    # Resolve path
    if hasattr(data_path, 'location'):
        resolved = data_path.location
    else:
        resolved = str(data_path)

    if task == "detect":
        yaml_path = os.path.join(resolved, "data.yaml") if os.path.isdir(resolved) else resolved
        if not os.path.exists(yaml_path):
            print(f"⏭️  {model_name}: {yaml_path} not found, SKIPPING")
            TRAINING_LOG.append((model_name, 'SKIPPED', 0, f'{yaml_path} not found'))
            return False
        resolved = yaml_path
    elif task == "classify":
        if not os.path.isdir(resolved):
            print(f"⏭️  {model_name}: {resolved} not a directory, SKIPPING")
            TRAINING_LOG.append((model_name, 'SKIPPED', 0, f'{resolved} not a directory'))
            return False

    # Determine base model — allows loading v4 models for fine-tuning
    if base_model is None:
        base_model = "yolov8m-cls.pt" if task == "classify" else "yolov8m.pt"
    attempts = [(batch, "full batch"), (max(batch // 2, 2), "half batch")]

    for batch_size, label in attempts:
        try:
            torch.cuda.empty_cache()
            gc.collect()
            print("=" * 60)
            print(f"🚀 Training {model_name} ({label}, batch={batch_size})")
            print(f"   Base: {base_model}")
            print(f"   Data: {resolved}")
            print(f"   Epochs: {epochs}, ImgSz: {imgsz}, Task: {task}")
            print("=" * 60)

            t0 = time.time()
            model = YOLO(base_model)
            model.train(
                data=resolved, epochs=epochs, imgsz=imgsz,
                batch=batch_size, name=model_name.replace('.pt', ''),
                device=0, **kwargs
            )
            elapsed = (time.time() - t0) / 60
            print(f"✅ {model_name} done in {elapsed:.1f} min")
            TRAINING_LOG.append((model_name, 'TRAINED', 0, f'{elapsed:.1f} min'))
            return True

        except RuntimeError as e:
            if 'out of memory' in str(e).lower() or 'CUDA' in str(e):
                print(f"⚠️  OOM on {label}, clearing cache...")
                torch.cuda.empty_cache()
                gc.collect()
                if label == "half batch":
                    TRAINING_LOG.append((model_name, 'OOM_FAIL', 0, str(e)[:100]))
                    return False
            else:
                TRAINING_LOG.append((model_name, 'ERROR', 0, str(e)[:100]))
                traceback.print_exc()
                return False
        except Exception as e:
            TRAINING_LOG.append((model_name, 'ERROR', 0, str(e)[:100]))
            traceback.print_exc()
            return False
    return False


def download_model(model_name, min_map50=0.3, task="classify"):
    """Validate, copy, and download a trained YOLO model."""
    from ultralytics import YOLO
    base = model_name.replace('.pt', '')

    search_dirs = ["runs/classify", "runs/detect"]
    paths = []
    for sd in search_dirs:
        paths.extend(sorted(glob.glob(f"{sd}/{base}*/weights/best.pt")))

    if not paths:
        print(f"❌ {model_name}: no training run found")
        TRAINING_LOG.append((model_name, 'MISSING', 0, 'no training run'))
        return

    best_path = paths[-1]
    print(f"📦 Found: {best_path}")

    try:
        model = YOLO(best_path)
        metrics = model.val()
        if task == "classify":
            score = getattr(metrics, 'top1', 0) or 0
            metric_name = "top1_acc"
        else:
            score = metrics.box.map50 if hasattr(metrics, 'box') else 0
            metric_name = "mAP50"
        print(f"   {metric_name}: {score:.4f}")
        if score >= min_map50:
            TRAINING_LOG.append((model_name, 'PASS', score, f'{metric_name}={score:.4f}'))
        else:
            print(f"   ⚠️  Below threshold — downloading anyway")
            TRAINING_LOG.append((model_name, 'LOW_MAP', score, f'{metric_name}={score:.4f}'))
    except Exception as e:
        print(f"   ⚠️  Validation failed: {e}")
        TRAINING_LOG.append((model_name, 'DOWNLOADED (no val)', 0, str(e)[:80]))

    shutil.copy(best_path, model_name)
    files.download(model_name)
    print(f"✅ Downloaded {model_name}")


def download_hf_model(model_dir, zip_name):
    """Zip and download a HuggingFace model directory."""
    if not os.path.isdir(model_dir):
        print(f"❌ {model_dir} not found")
        TRAINING_LOG.append((zip_name, 'MISSING', 0, 'directory not found'))
        return
    shutil.make_archive(zip_name.replace('.zip', ''), 'zip', model_dir)
    files.download(zip_name)
    print(f"✅ Downloaded {zip_name}")
    TRAINING_LOG.append((zip_name, 'PASS', 0, 'downloaded'))


print("✅ All helper functions defined")
print("   safe_download_video, safe_download_video_segment,")
print("   extract_frames_from_clip, extract_frames_as_images,")
print("   label_clip_with_claude, safe_train, download_model, download_hf_model")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 0D — Sport Label Configs + Directory Structure
# ═══════════════════════════════════════════════════════
SPORT_LABELS = {
    "basketball": [
        "made_shot", "missed_shot", "dunk", "layup", "three_pointer",
        "rebound", "steal", "block", "fast_break", "assist",
        "turnover", "dead_ball"
    ],
    "football": [
        "pass_complete", "pass_incomplete", "touchdown", "interception",
        "sack", "run_play", "scramble", "reception_yac", "fumble", "dead_ball"
    ],
    "lacrosse": [
        "goal", "shot_on_goal", "save", "ground_ball",
        "face_off", "dodge", "clear", "dead_ball"
    ]
}
SPORTS = list(SPORT_LABELS.keys())

# Directory structure
BASE_DIR = "/content"
RAW_VIDEO_DIR = os.path.join(BASE_DIR, "raw_videos")
CLIPS_DIR = os.path.join(BASE_DIR, "clips")
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
MODELS_DIR = os.path.join(BASE_DIR, "models")
FRAMES_DIR = os.path.join(BASE_DIR, "frames_dataset")
FEEDBACK_DIR = os.path.join(BASE_DIR, "feedback_dataset")

for d in [RAW_VIDEO_DIR, CLIPS_DIR, DATASET_DIR, MODELS_DIR, FRAMES_DIR, FEEDBACK_DIR]:
    os.makedirs(d, exist_ok=True)

print("Sport labels configured:")
for sport, labels in SPORT_LABELS.items():
    print(f"  {sport}: {len(labels)} classes — {', '.join(labels)}")
print(f"\nDirectories created under {BASE_DIR}/")

---
## SECTION 1: YouTube Highlight Harvester

Downloads highlight compilation videos from YouTube, splits them into individual clips
using scene detection, and filters to the 2-15 second sweet spot.

**How it works:**
1. `yt-dlp` downloads each video at 720p (uses YouTube search queries to find compilations automatically)
2. PySceneDetect finds scene boundaries (cuts between highlights)
3. Clips outside 2-15s are filtered out
4. Quality gate verifies minimum clip counts with duration histogram

**You can:** Add/replace YouTube URLs or modify the search queries below for more data.

In [ ]:
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# Cell 1B — Highlight Playlist URLs per Sport
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# ~30 videos per sport for robust training data.

HIGHLIGHT_URLS = {
    "basketball": [
        "ytsearch5:NBA top 10 plays of the week 2024",
        "ytsearch5:NBA best dunks compilation 2024",
        "ytsearch3:NBA best three pointers compilation 2024",
        "ytsearch3:NBA steals and blocks highlights 2024",
        "ytsearch3:NBA best assists compilation 2024",
        "ytsearch3:NBA fast break highlights compilation 2024",
        "ytsearch3:NBA best layups compilation",
        "ytsearch3:college basketball march madness highlights 2024",
        "ytsearch3:NCAA basketball tournament best plays 2024",
        "ytsearch3:NBA game film full court camera 2024",
        "ytsearch2:NBA sideline camera highlights jersey",
    ],
    "football": [
        "ytsearch5:NFL top 10 plays of the week 2024",
        "ytsearch5:NFL best touchdowns compilation 2024",
        "ytsearch3:NFL greatest catches all time",
        "ytsearch3:NFL best interceptions compilation 2024",
        "ytsearch3:NFL biggest sacks and hits 2024",
        "ytsearch3:NFL best runs and scrambles compilation 2024",
        "ytsearch3:NFL quarterback highlights compilation 2024",
        "ytsearch3:college football highlights 2024",
        "ytsearch3:high school football highlights hudl 2024",
        "ytsearch3:NFL game film all-22 camera 2024",
        "ytsearch2:football sideline camera jersey number",
    ],
    "lacrosse": [
        "ytsearch5:PLL lacrosse highlights 2024",
        "ytsearch5:best lacrosse goals compilation 2024",
        "ytsearch3:lacrosse best saves compilation",
        "ytsearch3:lacrosse face off highlights 2024",
        "ytsearch3:MLL lacrosse top plays",
        "ytsearch3:NLL lacrosse highlights 2024",
        "ytsearch3:college lacrosse highlights 2024",
        "ytsearch3:NCAA lacrosse championship highlights",
        "ytsearch3:high school lacrosse highlights 2024",
        "ytsearch2:lacrosse game film sideline camera",
    ],
}

for sport, urls in HIGHLIGHT_URLS.items():
    total_vids = sum(int(u.split(":")[0].replace("ytsearch", "") or "1")
                     for u in urls if u.startswith("ytsearch"))
    print(f"{sport}: {len(urls)} queries \u2192 ~{total_vids} videos")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 1C — Download Videos with yt-dlp
# ═══════════════════════════════════════════════════════
print("=" * 60)
print("🎬 Downloading highlight videos...")
print("=" * 60)

download_stats = {}
for sport in SPORTS:
    sport_dir = os.path.join(RAW_VIDEO_DIR, sport)
    urls = HIGHLIGHT_URLS.get(sport, [])

    if not urls:
        print(f"\n⏭️  {sport}: No URLs configured")
        download_stats[sport] = {"attempted": 0, "downloaded": 0}
        continue

    print(f"\n{'─' * 40}")
    print(f"📥 {sport.upper()}: {len(urls)} sources")
    downloaded = 0

    for i, url in enumerate(urls):
        label = url[:60] if url.startswith("https://") else url.split(":", 1)[-1][:50]
        print(f"  [{i+1}/{len(urls)}] {label}...")

        # For search queries, yt-dlp handles ytsearchN: natively
        result = safe_download_video(url, sport_dir)
        if result:
            size_mb = os.path.getsize(result) / 1e6
            print(f"    ✅ {os.path.basename(result)} ({size_mb:.1f} MB)")
            downloaded += 1
        else:
            print(f"    ❌ Failed")

    # Count all downloaded videos (some search queries download multiple)
    all_vids = glob.glob(os.path.join(sport_dir, "*.mp4"))
    download_stats[sport] = {"attempted": len(urls), "downloaded": len(all_vids)}

print("\n" + "=" * 60)
print("Download Summary:")
for sport, stats in download_stats.items():
    print(f"  {sport}: {stats['downloaded']} videos downloaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 1D — Scene Detection: Split Videos into Clips
# ═══════════════════════════════════════════════════════
from scenedetect import detect, ContentDetector

MIN_CLIP_DURATION = 2.0   # seconds
MAX_CLIP_DURATION = 15.0  # seconds

print("=" * 60)
print("✂️  Splitting videos into clips via scene detection...")
print("=" * 60)

clip_stats = {}
for sport in SPORTS:
    sport_raw = os.path.join(RAW_VIDEO_DIR, sport)
    sport_clips = os.path.join(CLIPS_DIR, sport)
    os.makedirs(sport_clips, exist_ok=True)

    videos = sorted(glob.glob(os.path.join(sport_raw, "*.mp4")))
    if not videos:
        print(f"\n⏭️  {sport}: No videos found")
        clip_stats[sport] = {"total": 0, "kept": 0, "filtered": 0}
        continue

    print(f"\n{'─' * 40}")
    print(f"🔍 {sport.upper()}: {len(videos)} videos")
    total_scenes = 0
    kept = 0
    clip_idx = len(glob.glob(os.path.join(sport_clips, "*.mp4")))  # Resume-safe

    for video_path in videos:
        vname = os.path.basename(video_path)
        print(f"  Scanning {vname}...")
        try:
            scene_list = detect(video_path, ContentDetector(threshold=25.0))
            total_scenes += len(scene_list)

            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            for start, end in scene_list:
                sf = start.get_frames()
                ef = end.get_frames()
                duration = (ef - sf) / fps
                if not (MIN_CLIP_DURATION <= duration <= MAX_CLIP_DURATION):
                    continue

                clip_idx += 1
                out_path = os.path.join(sport_clips, f"clip_{clip_idx:04d}.mp4")
                if os.path.exists(out_path):
                    kept += 1
                    continue

                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

                cap.set(cv2.CAP_PROP_POS_FRAMES, sf)
                for _ in range(ef - sf):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    writer.write(frame)
                writer.release()
                kept += 1

            cap.release()
            print(f"    → {len(scene_list)} scenes, {kept} clips so far")
        except Exception as e:
            print(f"    ❌ Error: {e}")

    filtered = total_scenes - kept
    clip_stats[sport] = {"total": total_scenes, "kept": kept, "filtered": filtered}

print("\n" + "=" * 60)
print("Scene Detection Summary:")
for sport, s in clip_stats.items():
    print(f"  {sport}: {s['kept']} clips kept (from {s['total']} scenes, {s['filtered']} filtered)")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 1E — Quality Gate + Duration Histogram
# ═══════════════════════════════════════════════════════
MIN_CLIPS_PER_SPORT = 100

print("=" * 60)
print("🔍 Quality Gate: Checking clip counts")
print("=" * 60)

all_clear = True
all_durations = {}

for sport in SPORTS:
    clips = sorted(glob.glob(os.path.join(CLIPS_DIR, sport, "*.mp4")))
    count = len(clips)

    if count == 0:
        print(f"  ❌ {sport}: 0 clips — add YouTube URLs and re-run Section 1")
        all_clear = False
    elif count < MIN_CLIPS_PER_SPORT:
        print(f"  ⚠️  {sport}: {count} clips — below {MIN_CLIPS_PER_SPORT} target")
        all_clear = False
    else:
        print(f"  ✅ {sport}: {count} clips")

    # Collect durations
    durations = []
    for cp in clips[:500]:
        cap = cv2.VideoCapture(cp)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        nf = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        if fps > 0 and nf > 0:
            durations.append(nf / fps)
    if durations:
        all_durations[sport] = durations
        print(f"    Duration: avg={sum(durations)/len(durations):.1f}s, "
              f"min={min(durations):.1f}s, max={max(durations):.1f}s")

# ── Duration histogram ──
if all_durations:
    fig, axes = plt.subplots(1, len(all_durations), figsize=(5 * len(all_durations), 4))
    if len(all_durations) == 1:
        axes = [axes]
    for ax, (sport, durs) in zip(axes, all_durations.items()):
        ax.hist(durs, bins=20, color='#00A3FF', edgecolor='white', alpha=0.8)
        ax.set_title(f"{sport.capitalize()} ({len(durs)} clips)")
        ax.set_xlabel("Duration (seconds)")
        ax.set_ylabel("Count")
        ax.axvline(x=sum(durs)/len(durs), color='red', linestyle='--',
                    label=f"avg={sum(durs)/len(durs):.1f}s")
        ax.legend()
    plt.suptitle("Clip Duration Distribution", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if all_clear:
    print("\n✅ All sports have sufficient clips. Proceed to Section 2.")
else:
    print("\n⚠️  Some sports are low. Training will still work but accuracy may suffer.")

---
## SECTION 2: Claude Vision Auto-Labeling

Uses Claude Sonnet to classify each clip by extracting 4 evenly-spaced frames
and asking the model to identify the basketball/football/lacrosse play outcome.

**Cost estimate:** ~$0.012 per clip (4 images × ~$0.003 each)
- 500 clips ≈ $6
- 1000 clips ≈ $12

**Resume-safe:** Labels are saved incrementally to `labels_{sport}.json`.
If Colab disconnects, re-running this section skips already-labeled clips.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2B — Batch Labeling Function
# ═══════════════════════════════════════════════════════
def label_all_clips(sport, resume=True):
    """Label all clips for a sport using Claude Vision. Resume-safe."""
    sport_clips_dir = os.path.join(CLIPS_DIR, sport)
    labels_file = os.path.join(CLIPS_DIR, f"labels_{sport}.json")
    labels = SPORT_LABELS[sport]

    existing = {}
    if resume and os.path.exists(labels_file):
        with open(labels_file) as f:
            existing = json.load(f)
        print(f"  Resuming: {len(existing)} clips already labeled")

    clips = sorted(glob.glob(os.path.join(sport_clips_dir, "*.mp4")))
    if not clips:
        print(f"  No clips found for {sport}")
        return existing

    to_label = [c for c in clips if os.path.basename(c) not in existing]
    print(f"  {len(to_label)} clips to label ({len(existing)} already done)")

    for i, clip_path in enumerate(to_label):
        clip_name = os.path.basename(clip_path)
        result = label_clip_with_claude(clip_path, sport, labels)
        existing[clip_name] = result

        # Save incrementally every 5 clips
        if (i + 1) % 5 == 0 or i == len(to_label) - 1:
            with open(labels_file, 'w') as f:
                json.dump(existing, f, indent=2)

        if (i + 1) % 25 == 0:
            print(f"    [{i+1}/{len(to_label)}] {clip_name} → {result['label']} "
                  f"({result.get('confidence', 0):.2f})")

        # Rate limiting — 1 second between API calls
        time.sleep(1.0)

    # Final save
    with open(labels_file, 'w') as f:
        json.dump(existing, f, indent=2)
    return existing

print("✅ label_all_clips() defined")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2C — Run Auto-Labeling for All Sports
# ═══════════════════════════════════════════════════════
print("=" * 60)
print("🏷️  Claude Vision Auto-Labeling")
print("=" * 60)

total_clips = sum(len(glob.glob(os.path.join(CLIPS_DIR, s, "*.mp4"))) for s in SPORTS)
est_cost = total_clips * 0.012
print(f"Total clips: {total_clips}")
print(f"Estimated Claude API cost: ~${est_cost:.2f}\n")

all_labels = {}
for sport in SPORTS:
    clips = glob.glob(os.path.join(CLIPS_DIR, sport, "*.mp4"))
    if not clips:
        print(f"\n⏭️  {sport}: No clips, skipping")
        continue

    print(f"\n{'─' * 40}")
    print(f"🏷️  {sport.upper()} ({len(clips)} clips)")

    sport_labels = label_all_clips(sport)
    all_labels[sport] = sport_labels

    if sport_labels:
        counts = Counter(v['label'] for v in sport_labels.values())
        print(f"\n  Label distribution:")
        for label, count in counts.most_common():
            pct = count / len(sport_labels) * 100
            bar = "█" * int(pct / 2)
            print(f"    {label:25s} {count:4d} ({pct:5.1f}%) {bar}")

        unknowns = sum(1 for v in sport_labels.values() if v['label'] == 'unknown')
        low_conf = sum(1 for v in sport_labels.values()
                       if v.get('confidence', 0) < 0.5 and v['label'] != 'unknown')
        if unknowns:
            print(f"  ⚠️  {unknowns} clips labeled 'unknown'")
        if low_conf:
            print(f"  ⚠️  {low_conf} clips with confidence < 0.5")

print("\n" + "=" * 60)
print("✅ Labeling complete!")
for sport, labels in all_labels.items():
    print(f"  {sport}: {len(labels)} clips labeled")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2D — Build Video Classification Dataset
# ═══════════════════════════════════════════════════════
TRAIN_SPLIT = 0.8
MIN_CLASS_SIZE = 5

print("=" * 60)
print("📂 Building classification datasets (80/20 train/val)")
print("=" * 60)

dataset_info = {}
for sport in SPORTS:
    labels_file = os.path.join(CLIPS_DIR, f"labels_{sport}.json")
    if not os.path.exists(labels_file):
        print(f"\n⏭️  {sport}: No labels file, skipping")
        continue

    with open(labels_file) as f:
        labels = json.load(f)

    by_label = defaultdict(list)
    for clip_name, info in labels.items():
        lbl = info.get('label', 'unknown')
        if lbl == 'unknown':
            continue
        if info.get('confidence', 0) < 0.3:
            continue
        by_label[lbl].append(clip_name)

    # Merge tiny classes into "other"
    final_labels = {}
    other_clips = []
    for lbl, clip_list in by_label.items():
        if len(clip_list) >= MIN_CLASS_SIZE:
            final_labels[lbl] = clip_list
        else:
            other_clips.extend(clip_list)
    if other_clips:
        final_labels["other"] = other_clips

    print(f"\n{'─' * 40}")
    print(f"📂 {sport.upper()}: {len(final_labels)} classes")

    sport_ds = os.path.join(DATASET_DIR, sport)
    total_train = 0
    total_val = 0
    for lbl, clip_list in sorted(final_labels.items()):
        random.shuffle(clip_list)
        split = max(int(len(clip_list) * TRAIN_SPLIT), 1)
        train_clips = clip_list[:split]
        val_clips = clip_list[split:] if split < len(clip_list) else clip_list[-1:]

        for split_name, split_clips in [("train", train_clips), ("val", val_clips)]:
            split_dir = os.path.join(sport_ds, split_name, lbl)
            os.makedirs(split_dir, exist_ok=True)
            for cn in split_clips:
                src = os.path.join(CLIPS_DIR, sport, cn)
                if os.path.exists(src):
                    shutil.copy2(src, os.path.join(split_dir, cn))

        total_train += len(train_clips)
        total_val += len(val_clips)
        print(f"  {lbl:25s} train={len(train_clips):4d}  val={len(val_clips):4d}")

    dataset_info[sport] = {
        "classes": list(final_labels.keys()),
        "total": total_train + total_val,
        "train": total_train, "val": total_val
    }

print("\n" + "=" * 60)
print("Dataset Summary:")
for sport, info in dataset_info.items():
    print(f"  {sport}: {info['total']} clips, {len(info['classes'])} classes "
          f"(train={info['train']}, val={info['val']})")

---
## SECTION 3: VideoMAE Training — Basketball

Fine-tunes a VideoMAE model (pretrained on Kinetics-400) for basketball outcome classification.
VideoMAE understands temporal sequences — it sees 16 frames from each clip to classify the action.

**Model:** `MCG-NJU/videomae-base-finetuned-kinetics` → fine-tuned with basketball labels
**Output:** `videomae_basketball_v5/` directory (model + processor + ONNX export)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3A — VideoMAE Setup + Training Function
# ═══════════════════════════════════════════════════════
from transformers import (
    VideoMAEForVideoClassification, VideoMAEImageProcessor,
    TrainingArguments, Trainer
)
import evaluate
from torch.utils.data import Dataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

VIDEOMAE_MODEL_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"
NUM_FRAMES = 16

# ── Video Dataset ──
class VideoClipDataset(Dataset):
    """Loads video clips from folder structure: split/label/clip.mp4"""

    def __init__(self, root_dir, processor, num_frames=16, is_train=True):
        self.processor = processor
        self.num_frames = num_frames
        self.is_train = is_train
        self.samples = []  # (path, label_idx)
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_dir):
                if fname.endswith('.mp4'):
                    self.samples.append((os.path.join(cls_dir, fname),
                                         self.class_to_idx[cls_name]))

        if is_train:
            random.shuffle(self.samples)
        print(f"    Loaded {len(self.samples)} clips, {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = self._load_video(path)

        # Apply augmentation for training
        if self.is_train:
            # Random horizontal flip
            if random.random() > 0.5:
                frames = [np.fliplr(f).copy() for f in frames]

        inputs = self.processor(list(frames), return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return {"pixel_values": pixel_values, "labels": label}

    def _load_video(self, path):
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            total = 1
        indices = np.linspace(0, total - 1, self.num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Center crop to square
                h, w = frame.shape[:2]
                if w > h:
                    offset = (w - h) // 2
                    frame = frame[:, offset:offset+h]
                elif h > w:
                    offset = (h - w) // 2
                    frame = frame[offset:offset+w, :]
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
            elif frames:
                frames.append(frames[-1].copy())
            else:
                frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
        cap.release()
        while len(frames) < self.num_frames:
            frames.append(frames[-1].copy() if frames else np.zeros((224, 224, 3), dtype=np.uint8))
        return frames[:self.num_frames]


def train_videomae(sport, epochs=15, batch_size=4, lr=5e-5):
    """Train VideoMAE for a sport. Returns output directory or None."""
    train_dir = os.path.join(DATASET_DIR, sport, "train")
    val_dir = os.path.join(DATASET_DIR, sport, "val")

    if not os.path.isdir(train_dir):
        print(f"⏭️  {sport}: No training data at {train_dir}")
        TRAINING_LOG.append((f"videomae_{sport}_v5", 'SKIPPED', 0, 'no data'))
        return None

    print("=" * 60)
    print(f"🧠 VideoMAE Training: {sport.upper()}")
    print("=" * 60)

    processor = VideoMAEImageProcessor.from_pretrained(VIDEOMAE_MODEL_NAME)
    print("  Loading training data...")
    train_ds = VideoClipDataset(train_dir, processor, NUM_FRAMES, is_train=True)
    print("  Loading validation data...")
    val_ds = VideoClipDataset(val_dir, processor, NUM_FRAMES, is_train=False)

    num_classes = len(train_ds.classes)
    label2id = {c: i for i, c in enumerate(train_ds.classes)}
    id2label = {i: c for c, i in label2id.items()}
    print(f"  Classes ({num_classes}): {train_ds.classes}")

    model = VideoMAEForVideoClassification.from_pretrained(
        VIDEOMAE_MODEL_NAME,
        num_labels=num_classes,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True
    )

    output_dir = os.path.join(MODELS_DIR, f"videomae_{sport}_v5")
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=10,
        remove_unused_columns=False,
        report_to="none",
    )

    acc_metric = evaluate.load("accuracy")
    def compute_metrics(pred):
        preds = pred.predictions.argmax(-1)
        return acc_metric.compute(predictions=preds, references=pred.label_ids)

    def _do_train(trainer_obj, t0):
        trainer_obj.train()
        elapsed = (time.time() - t0) / 60
        results = trainer_obj.evaluate()
        acc = results.get("eval_accuracy", 0)
        print(f"\n✅ {sport} VideoMAE done in {elapsed:.1f} min — accuracy: {acc:.4f}")
        TRAINING_LOG.append((f"videomae_{sport}_v5", 'TRAINED', acc, f'{elapsed:.1f} min, acc={acc:.4f}'))

        # ── Confusion Matrix ──
        print("  Computing confusion matrix...")
        preds_output = trainer_obj.predict(val_ds)
        y_pred = preds_output.predictions.argmax(-1)
        y_true = [val_ds[i]["labels"] for i in range(len(val_ds))]
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=train_ds.classes)
        fig, ax = plt.subplots(figsize=(max(8, num_classes), max(6, num_classes * 0.7)))
        disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
        ax.set_title(f"VideoMAE {sport.capitalize()} — Confusion Matrix (acc={acc:.3f})")
        plt.tight_layout()
        plt.show()

        # Save model + processor + class mapping
        trainer_obj.save_model(output_dir)
        processor.save_pretrained(output_dir)
        with open(os.path.join(output_dir, "classes.json"), 'w') as f:
            json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)

        # ── ONNX Export ──
        print("  Exporting to ONNX...")
        try:
            onnx_path = os.path.join(output_dir, "model.onnx")
            dummy_input = torch.randn(1, NUM_FRAMES, 3, 224, 224).to(model.device)
            torch.onnx.export(
                model, dummy_input, onnx_path,
                input_names=["pixel_values"],
                output_names=["logits"],
                dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
                opset_version=14,
            )
            onnx_size = os.path.getsize(onnx_path) / 1e6
            print(f"  ✅ ONNX exported: {onnx_path} ({onnx_size:.1f} MB)")
        except Exception as e:
            print(f"  ⚠️  ONNX export failed (model still saved as PyTorch): {e}")

        return output_dir

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    try:
        return _do_train(trainer, t0)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and batch_size > 2:
            print(f"⚠️  OOM with batch={batch_size}, retrying with batch=2...")
            torch.cuda.empty_cache()
            gc.collect()
            training_args.per_device_train_batch_size = 2
            training_args.per_device_eval_batch_size = 2
            model2 = VideoMAEForVideoClassification.from_pretrained(
                VIDEOMAE_MODEL_NAME, num_labels=num_classes,
                label2id=label2id, id2label=id2label,
                ignore_mismatched_sizes=True
            )
            trainer2 = Trainer(model=model2, args=training_args,
                               train_dataset=train_ds, eval_dataset=val_ds,
                               compute_metrics=compute_metrics)
            try:
                return _do_train(trainer2, t0)
            except Exception as e2:
                TRAINING_LOG.append((f"videomae_{sport}_v5", 'OOM_FAIL', 0, str(e2)[:100]))
                traceback.print_exc()
                return None
        else:
            TRAINING_LOG.append((f"videomae_{sport}_v5", 'ERROR', 0, str(e)[:100]))
            traceback.print_exc()
            return None
    except Exception as e:
        TRAINING_LOG.append((f"videomae_{sport}_v5", 'ERROR', 0, str(e)[:100]))
        traceback.print_exc()
        return None


print("✅ VideoMAE training function + VideoClipDataset defined")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3B — Train VideoMAE: Basketball
# ═══════════════════════════════════════════════════════
bball_videomae_dir = train_videomae("basketball", epochs=25, batch_size=4, lr=5e-5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3C — Download VideoMAE: Basketball
# ═══════════════════════════════════════════════════════
bball_dir = os.path.join(MODELS_DIR, "videomae_basketball_v5")
download_hf_model(bball_dir, "videomae_basketball_v5.zip")

---
## SECTION 4: VideoMAE Training — Football

Same architecture as Section 3, trained on football outcome labels.
Reuses `train_videomae()` and `VideoClipDataset` defined in Cell 3A.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4A — VideoMAE Setup (Football)
# ═══════════════════════════════════════════════════════
# Model: MCG-NJU/videomae-base-finetuned-kinetics
# Classes: pass_complete, pass_incomplete, touchdown, interception,
#          sack, run_play, scramble, reception_yac, fumble, dead_ball
# Training function: train_videomae() from Cell 3A
# Dataset: /content/dataset/football/train/ and val/

football_train = os.path.join(DATASET_DIR, "football", "train")
if os.path.isdir(football_train):
    classes = sorted(os.listdir(football_train))
    total = sum(len(os.listdir(os.path.join(football_train, c)))
                for c in classes if os.path.isdir(os.path.join(football_train, c)))
    print(f"Football dataset: {total} clips across {len(classes)} classes")
    for c in classes:
        cdir = os.path.join(football_train, c)
        if os.path.isdir(cdir):
            print(f"  {c}: {len(os.listdir(cdir))} clips")
else:
    print("⚠️  No football training data found — Cell 4B will skip")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4B — Train VideoMAE: Football
# ═══════════════════════════════════════════════════════
football_videomae_dir = train_videomae("football", epochs=25, batch_size=4, lr=5e-5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4C — Download VideoMAE: Football
# ═══════════════════════════════════════════════════════
fb_dir = os.path.join(MODELS_DIR, "videomae_football_v5")
download_hf_model(fb_dir, "videomae_football_v5.zip")

---
## SECTION 5: VideoMAE Training — Lacrosse

Same architecture as Section 3, trained on lacrosse outcome labels.
Reuses `train_videomae()` and `VideoClipDataset` defined in Cell 3A.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5A — VideoMAE Setup (Lacrosse)
# ═══════════════════════════════════════════════════════
lacrosse_train = os.path.join(DATASET_DIR, "lacrosse", "train")
if os.path.isdir(lacrosse_train):
    classes = sorted(os.listdir(lacrosse_train))
    total = sum(len(os.listdir(os.path.join(lacrosse_train, c)))
                for c in classes if os.path.isdir(os.path.join(lacrosse_train, c)))
    print(f"Lacrosse dataset: {total} clips across {len(classes)} classes")
    for c in classes:
        cdir = os.path.join(lacrosse_train, c)
        if os.path.isdir(cdir):
            print(f"  {c}: {len(os.listdir(cdir))} clips")
else:
    print("⚠️  No lacrosse training data found — Cell 5B will skip")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5B — Train VideoMAE: Lacrosse
# ═══════════════════════════════════════════════════════
lacrosse_videomae_dir = train_videomae("lacrosse", epochs=25, batch_size=4, lr=5e-5)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5C — Download VideoMAE: Lacrosse
# ═══════════════════════════════════════════════════════
lax_dir = os.path.join(MODELS_DIR, "videomae_lacrosse_v5")
download_hf_model(lax_dir, "videomae_lacrosse_v5.zip")

---
## SECTION 6: YOLO Outcome Classifier

Converts Claude-labeled clips into frame-level image classification datasets,
then trains YOLOv8m-cls models for each sport. These complement VideoMAE by
providing fast frame-level predictions that can run alongside v4 object detectors.

**3 models:** one per sport, `yolov8m-cls` base, 75 epochs each.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6A — Convert Clips to Frame-Level YOLO Dataset
# ═══════════════════════════════════════════════════════
FRAMES_PER_CLIP = 8  # Extract 8 frames per clip for training

print("=" * 60)
print("🖼️  Building frame-level classification datasets for YOLO")
print("=" * 60)

yolo_dataset_info = {}
for sport in SPORTS:
    labels_file = os.path.join(CLIPS_DIR, f"labels_{sport}.json")
    if not os.path.exists(labels_file):
        print(f"\n⏭️  {sport}: No labels, skipping")
        continue

    with open(labels_file) as f:
        labels = json.load(f)

    sport_frames = os.path.join(FRAMES_DIR, sport)
    frame_count = 0
    class_counts = defaultdict(int)

    for clip_name, info in labels.items():
        lbl = info.get('label', 'unknown')
        if lbl == 'unknown' or info.get('confidence', 0) < 0.3:
            continue

        clip_path = os.path.join(CLIPS_DIR, sport, clip_name)
        if not os.path.exists(clip_path):
            continue

        # Consistent train/val split by clip name hash
        split = "train" if hash(clip_name) % 5 != 0 else "val"
        out_dir = os.path.join(sport_frames, split, lbl)
        os.makedirs(out_dir, exist_ok=True)

        # Extract frames
        raw_frames = extract_frames_as_images(clip_path, num_frames=FRAMES_PER_CLIP)
        for fi, frame in enumerate(raw_frames):
            fname = f"{clip_name.replace('.mp4', '')}_f{fi:02d}.jpg"
            out_path = os.path.join(out_dir, fname)
            if not os.path.exists(out_path):
                cv2.imwrite(out_path, frame)
            frame_count += 1
            class_counts[lbl] += 1

    print(f"\n{'─' * 40}")
    print(f"🖼️  {sport.upper()}: {frame_count} frames extracted")
    for lbl, cnt in sorted(class_counts.items()):
        print(f"    {lbl:25s} {cnt:5d} frames")

    yolo_dataset_info[sport] = {"frames": frame_count, "classes": dict(class_counts)}

print("\n✅ Frame datasets ready for YOLO training")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6B — Train YOLO Outcome Classifier: Basketball
# ═══════════════════════════════════════════════════════
bball_frames = os.path.join(FRAMES_DIR, "basketball")
safe_train(
    "outcome_classifier_basketball_v5",
    bball_frames, epochs=100, imgsz=224, batch=16,
    task="classify", lr0=0.001, patience=15
)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6B-dl — Download outcome_classifier_basketball_v5
# ═══════════════════════════════════════════════════════
download_model("outcome_classifier_basketball_v5.pt", min_map50=0.3, task="classify")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6C — Train YOLO Outcome Classifier: Football
# ═══════════════════════════════════════════════════════
fb_frames = os.path.join(FRAMES_DIR, "football")
safe_train(
    "outcome_classifier_football_v5",
    fb_frames, epochs=100, imgsz=224, batch=16,
    task="classify", lr0=0.001, patience=15
)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6C-dl — Download outcome_classifier_football_v5
# ═══════════════════════════════════════════════════════
download_model("outcome_classifier_football_v5.pt", min_map50=0.3, task="classify")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6D — Train YOLO Outcome Classifier: Lacrosse
# ═══════════════════════════════════════════════════════
lax_frames = os.path.join(FRAMES_DIR, "lacrosse")
safe_train(
    "outcome_classifier_lacrosse_v5",
    lax_frames, epochs=100, imgsz=224, batch=16,
    task="classify", lr0=0.001, patience=15
)

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6D-dl — Download outcome_classifier_lacrosse_v5
# ═══════════════════════════════════════════════════════
download_model("outcome_classifier_lacrosse_v5.pt", min_map50=0.3, task="classify")

---
## SECTION 7: Jersey Number OCR v5 + Player Detector v5

Uses the YouTube clips we already downloaded to train:
1. **jersey_ocr_universal_v5.pt** — universal jersey number detector (YOLO, replaces v1/v2/v3 OCR)
2. **player_detector_v5.pt** — player bounding box detector (YOLO)

Claude Vision auto-labels bounding boxes around jersey numbers and players.
This is the **most critical** model for the pipeline.


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7A — Auto-Label Jersey Numbers + Players with Claude Vision
# ═══════════════════════════════════════════════════════
OCR_FRAMES_DIR = os.path.join(BASE_DIR, "ocr_frames")
OCR_LABELS_DIR = os.path.join(BASE_DIR, "ocr_labels")
os.makedirs(OCR_FRAMES_DIR, exist_ok=True)
os.makedirs(OCR_LABELS_DIR, exist_ok=True)

FRAMES_PER_VIDEO_OCR = 20

def extract_ocr_frames(sport):
    """Extract frames from clips for jersey OCR labeling."""
    sport_clips = os.path.join(CLIPS_DIR, sport)
    sport_frames = os.path.join(OCR_FRAMES_DIR, sport)
    os.makedirs(sport_frames, exist_ok=True)
    clips = sorted(glob.glob(os.path.join(sport_clips, "*.mp4")))
    if not clips:
        return 0
    count = 0
    for clip_path in clips:
        cap = cv2.VideoCapture(clip_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            continue
        indices = np.linspace(0, total - 1, min(FRAMES_PER_VIDEO_OCR, total), dtype=int)
        clip_name = Path(clip_path).stem
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                out_path = os.path.join(sport_frames, f"{clip_name}_f{idx:04d}.jpg")
                if not os.path.exists(out_path):
                    cv2.imwrite(out_path, frame)
                    count += 1
        cap.release()
    return count


def label_frame_jersey_bbox(frame_path, max_retries=2):
    """Use Claude Vision to find jersey numbers and player bounding boxes."""
    img = cv2.imread(frame_path)
    if img is None:
        return []
    h, w = img.shape[:2]
    if w > 640:
        scale = 640 / w
        img = cv2.resize(img, (640, int(h * scale)))
        h, w = img.shape[:2]
    _, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    b64 = base64.b64encode(buf).decode("utf-8")

    prompt = (
        f"Image is {w}x{h} pixels. Find ALL visible jersey numbers.\n"
        f"For each, return the bounding box of the NUMBER REGION (not the full player).\n"
        f"Also return the bounding box of the FULL PLAYER body.\n"
        f"Respond with ONLY a JSON array:\n"
        f'[{{"number": 23, "num_x1": 100, "num_y1": 50, "num_x2": 140, "num_y2": 90, '
        f'"player_x1": 80, "player_y1": 20, "player_x2": 160, "player_y2": 250}}]\n'
        f"If no jersey numbers visible, return [].\n"
        f"Coordinates are in pixels (0,0 is top-left)."
    )

    for attempt in range(max_retries):
        try:
            resp = _claude_client.messages.create(
                model="claude-sonnet-4-6", max_tokens=500,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                    {"type": "text", "text": prompt}
                ]}]
            )
            text = resp.content[0].text.strip()
            match = re.search(r"\[.*\]", text, re.DOTALL)
            if match:
                detections = json.loads(match.group())
                results = []
                for det in detections:
                    num = det.get("number")
                    if num is None or not (0 <= num <= 99):
                        continue
                    nx1, ny1 = det.get("num_x1", 0), det.get("num_y1", 0)
                    nx2, ny2 = det.get("num_x2", 0), det.get("num_y2", 0)
                    ncx = ((nx1 + nx2) / 2) / w
                    ncy = ((ny1 + ny2) / 2) / h
                    nw = (nx2 - nx1) / w
                    nh = (ny2 - ny1) / h
                    px1, py1 = det.get("player_x1", 0), det.get("player_y1", 0)
                    px2, py2 = det.get("player_x2", 0), det.get("player_y2", 0)
                    pcx = ((px1 + px2) / 2) / w
                    pcy = ((py1 + py2) / 2) / h
                    pw = (px2 - px1) / w
                    ph = (py2 - py1) / h
                    ncx, ncy, nw, nh = [max(0, min(1, v)) for v in [ncx, ncy, nw, nh]]
                    pcx, pcy, pw, ph = [max(0, min(1, v)) for v in [pcx, pcy, pw, ph]]
                    if nw > 0.01 and nh > 0.01 and pw > 0.01 and ph > 0.01:
                        results.append({
                            "number": num,
                            "num_bbox": [ncx, ncy, nw, nh],
                            "player_bbox": [pcx, pcy, pw, ph],
                        })
                return results
            return []
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
    return []


print("=" * 60)
print("\U0001f522 Extracting frames for jersey OCR labeling...")
print("=" * 60)

for sport in SPORTS:
    count = extract_ocr_frames(sport)
    print(f"  {sport}: {count} new frames extracted")

# Label with Claude Vision
ocr_labels = {}
for sport in SPORTS:
    sport_frames = os.path.join(OCR_FRAMES_DIR, sport)
    labels_file = os.path.join(OCR_LABELS_DIR, f"ocr_labels_{sport}.json")

    existing = {}
    if os.path.exists(labels_file):
        with open(labels_file) as f:
            existing = json.load(f)

    frame_files = sorted(glob.glob(os.path.join(sport_frames, "*.jpg")))
    to_label = [f for f in frame_files if os.path.basename(f) not in existing]

    print(f"\n{sport.upper()}: {len(to_label)} frames to label ({len(existing)} done)")

    for i, fpath in enumerate(to_label):
        fname = os.path.basename(fpath)
        dets = label_frame_jersey_bbox(fpath)
        existing[fname] = dets

        if (i + 1) % 10 == 0 or i == len(to_label) - 1:
            with open(labels_file, "w") as f:
                json.dump(existing, f, indent=2)

        if (i + 1) % 50 == 0:
            det_count = sum(len(v) for v in existing.values())
            print(f"    [{i+1}/{len(to_label)}] Total detections so far: {det_count}")

        time.sleep(0.5)

    with open(labels_file, "w") as f:
        json.dump(existing, f, indent=2)

    det_count = sum(len(v) for v in existing.values())
    numbers_found = Counter()
    for dets in existing.values():
        for d in dets:
            numbers_found[d["number"]] += 1
    ocr_labels[sport] = existing

    print(f"  Total: {det_count} jersey detections across {len(existing)} frames")
    if numbers_found:
        top10 = numbers_found.most_common(10)
        print(f"  Top numbers: {top10}")

print("\n\u2705 Jersey OCR labeling complete!")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7B — Build YOLO Datasets for Jersey OCR + Player Detection
# ═══════════════════════════════════════════════════════
JERSEY_OCR_DS = os.path.join(BASE_DIR, "jersey_ocr_dataset")
PLAYER_DET_DS = os.path.join(BASE_DIR, "player_det_dataset")

def build_yolo_detection_datasets():
    """Convert Claude Vision labels to YOLO detection format."""
    for ds in [JERSEY_OCR_DS, PLAYER_DET_DS]:
        for split in ["train", "val"]:
            os.makedirs(os.path.join(ds, split, "images"), exist_ok=True)
            os.makedirs(os.path.join(ds, split, "labels"), exist_ok=True)

    total_ocr = 0
    total_player = 0

    for sport in SPORTS:
        labels_file = os.path.join(OCR_LABELS_DIR, f"ocr_labels_{sport}.json")
        if not os.path.exists(labels_file):
            continue
        with open(labels_file) as f:
            labels = json.load(f)

        frames_with_dets = [(fname, dets) for fname, dets in labels.items() if dets]
        random.shuffle(frames_with_dets)
        split_idx = int(len(frames_with_dets) * 0.85)
        train_frames = frames_with_dets[:split_idx]
        val_frames = frames_with_dets[split_idx:]

        for split_name, split_data in [("train", train_frames), ("val", val_frames)]:
            for fname, dets in split_data:
                src_img = os.path.join(OCR_FRAMES_DIR, sport, fname)
                if not os.path.exists(src_img):
                    continue

                base = f"{sport}_{fname}"
                base_no_ext = base.replace(".jpg", "")

                for ds in [JERSEY_OCR_DS, PLAYER_DET_DS]:
                    dst_img = os.path.join(ds, split_name, "images", base)
                    if not os.path.exists(dst_img):
                        shutil.copy2(src_img, dst_img)

                ocr_lines = []
                player_lines = []
                for det in dets:
                    num = det["number"]
                    if 0 <= num <= 99:
                        cx, cy, w, h = det["num_bbox"]
                        ocr_lines.append(f"{num} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        total_ocr += 1
                    pcx, pcy, pw, ph = det["player_bbox"]
                    player_lines.append(f"0 {pcx:.6f} {pcy:.6f} {pw:.6f} {ph:.6f}")
                    total_player += 1

                ocr_label_path = os.path.join(JERSEY_OCR_DS, split_name, "labels", base_no_ext + ".txt")
                with open(ocr_label_path, "w") as f:
                    f.write("\n".join(ocr_lines))

                player_label_path = os.path.join(PLAYER_DET_DS, split_name, "labels", base_no_ext + ".txt")
                with open(player_label_path, "w") as f:
                    f.write("\n".join(player_lines))

    # Write data.yaml
    ocr_names = {i: str(i) for i in range(100)}
    with open(os.path.join(JERSEY_OCR_DS, "data.yaml"), "w") as f:
        f.write(f"path: {JERSEY_OCR_DS}\ntrain: train/images\nval: val/images\nnc: 100\nnames: {json.dumps(ocr_names)}\n")

    with open(os.path.join(PLAYER_DET_DS, "data.yaml"), "w") as f:
        f.write(f"path: {PLAYER_DET_DS}\ntrain: train/images\nval: val/images\nnc: 1\nnames:\n  0: person\n")

    return total_ocr, total_player


total_ocr, total_player = build_yolo_detection_datasets()
print(f"Jersey OCR dataset: {total_ocr} number annotations")
print(f"Player detection dataset: {total_player} player annotations")

for ds_name, ds_path in [("Jersey OCR", JERSEY_OCR_DS), ("Player Det", PLAYER_DET_DS)]:
    for split in ["train", "val"]:
        imgs = len(glob.glob(os.path.join(ds_path, split, "images", "*")))
        lbls = len(glob.glob(os.path.join(ds_path, split, "labels", "*")))
        print(f"  {ds_name} {split}: {imgs} images, {lbls} labels")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7C — Train Jersey OCR v5 (Universal)
# ═══════════════════════════════════════════════════════
# Replaces v1/v2/v3 OCR models with a single better model
# trained on real YouTube game film auto-labeled by Claude Vision.

print("=" * 60)
print("\U0001f522 Training jersey_ocr_universal_v5 (YOLOv8m, 100 classes)")
print("=" * 60)

ocr_yaml = os.path.join(JERSEY_OCR_DS, "data.yaml")
if os.path.exists(ocr_yaml):
    safe_train(
        "jersey_ocr_universal_v5.pt",
        ocr_yaml,
        epochs=120,
        imgsz=640,
        batch=16,
        task="detect",
        base_model="yolov8m.pt",
        patience=20,
        lr0=0.005,
        lrf=0.01,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.1,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.3,
        degrees=5.0,
        translate=0.1,
        scale=0.3,
        flipud=0.0,
        fliplr=0.5,
    )
else:
    print("\u23ed\ufe0f  No OCR dataset found \u2014 skipping")
    TRAINING_LOG.append(("jersey_ocr_universal_v5.pt", "SKIPPED", 0, "no dataset"))


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7C-dl — Download jersey_ocr_universal_v5
# ═══════════════════════════════════════════════════════
download_model("jersey_ocr_universal_v5.pt", min_map50=0.3, task="detect")


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7D — Train Player Detector v5
# ═══════════════════════════════════════════════════════
# Single-class person detector trained on real game film.
# Replaces football_player_detector.pt and player_isolator_v3.pt.

print("=" * 60)
print("\U0001f3c3 Training player_detector_v5 (YOLOv8m, 1 class)")
print("=" * 60)

player_yaml = os.path.join(PLAYER_DET_DS, "data.yaml")
if os.path.exists(player_yaml):
    safe_train(
        "player_detector_v5.pt",
        player_yaml,
        epochs=80,
        imgsz=640,
        batch=16,
        task="detect",
        base_model="yolov8m.pt",
        patience=15,
        lr0=0.01,
        mosaic=1.0,
        mixup=0.1,
        hsv_h=0.015,
        hsv_s=0.4,
        hsv_v=0.3,
        degrees=3.0,
        scale=0.3,
        fliplr=0.5,
    )
else:
    print("\u23ed\ufe0f  No player dataset found \u2014 skipping")
    TRAINING_LOG.append(("player_detector_v5.pt", "SKIPPED", 0, "no dataset"))


In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7D-dl — Download player_detector_v5
# ═══════════════════════════════════════════════════════
download_model("player_detector_v5.pt", min_map50=0.3, task="detect")


---
## SECTION 8: Final Summary + Diagnostics

Complete report and autopilot diagnostics.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 8A — Final Report
# ═══════════════════════════════════════════════════════
print("=" * 60)
print("📋 V5 AUTO-LABEL TRAINING REPORT")
print("=" * 60)

# Expected models
expected_models = [
    ("videomae_basketball_v5.zip", "VideoMAE Basketball"),
    ("videomae_football_v5.zip", "VideoMAE Football"),
    ("videomae_lacrosse_v5.zip", "VideoMAE Lacrosse"),
    ("outcome_classifier_basketball_v5.pt", "YOLO Basketball"),
    ("outcome_classifier_football_v5.pt", "YOLO Football"),
    ("outcome_classifier_lacrosse_v5.pt", "YOLO Lacrosse"),
]

print("\n─── Models ───")
downloaded = 0
for fname, label in expected_models:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / 1e6
        print(f"  ✅ {label:30s} → {fname} ({size_mb:.1f} MB)")
        downloaded += 1
    elif os.path.isdir(fname.replace('.zip', '').replace('.pt', '')):
        print(f"  📦 {label:30s} → trained but not yet downloaded")
    else:
        print(f"  ❌ {label:30s} → MISSING")

# Feedback models (conditional)
fb_models = sorted(glob.glob("outcome_classifier_*_feedback*.pt"))
for fm in fb_models:
    size_mb = os.path.getsize(fm) / 1e6
    print(f"  🔧 {os.path.basename(fm):30s} (feedback fine-tune, {size_mb:.1f} MB)")
    downloaded += 1

print(f"\n─── Dataset Stats ───")
for sport in SPORTS:
    clips = len(glob.glob(os.path.join(CLIPS_DIR, sport, "*.mp4")))
    labels_file = os.path.join(CLIPS_DIR, f"labels_{sport}.json")
    labeled = 0
    if os.path.exists(labels_file):
        with open(labels_file) as f:
            labeled = len(json.load(f))
    frames = 0
    for root, dirs, fls in os.walk(os.path.join(FRAMES_DIR, sport)):
        frames += len([f for f in fls if f.endswith(('.jpg', '.png'))])
    fb_frames = 0
    for root, dirs, fls in os.walk(os.path.join(FEEDBACK_DIR, sport)):
        fb_frames += len([f for f in fls if f.endswith(('.jpg', '.png'))])
    print(f"  {sport}: {clips} clips, {labeled} labeled, {frames} YOLO frames, {fb_frames} feedback frames")

# Claude API cost estimate
total_labeled = sum(
    len(json.load(open(os.path.join(CLIPS_DIR, f"labels_{s}.json"))))
    for s in SPORTS
    if os.path.exists(os.path.join(CLIPS_DIR, f"labels_{s}.json"))
)
est_cost = total_labeled * 0.012
print(f"\n─── Claude API Cost ───")
print(f"  Clips labeled: {total_labeled}")
print(f"  Estimated cost: ~${est_cost:.2f}")

print(f"\n─── Training Log ({len(TRAINING_LOG)} entries) ───")
for name, status, score, detail in TRAINING_LOG:
    icon = {"TRAINED": "✅", "PASS": "✅", "SKIPPED": "⏭️", "ERROR": "❌",
            "OOM_FAIL": "💥", "MISSING": "❌", "LOW_MAP": "⚠️"}.get(status, "📦")
    print(f"  {icon} {name:45s} {status:20s} {detail}")

print(f"\n─── Summary ───")
total_expected = len(expected_models) + len(fb_models)
print(f"  Models downloaded: {downloaded}/{total_expected}")
errors = [x for x in TRAINING_LOG if x[1] in ('ERROR', 'OOM_FAIL')]
if errors:
    print(f"  ⚠️  {len(errors)} training errors — check log above")
else:
    print(f"  ✅ No training errors")

print(f"\n─── Next Steps ───")
print("  1. Upload VideoMAE .zip models to your inference server")
print("  2. Upload YOLO .pt models to playerJerseyIdentification-master/app/model/")
print("  3. Update detection pipeline to use v5 outcome classifiers")
print("  4. Re-run this notebook with more YouTube URLs for better accuracy")
print("  5. After users review more clips, re-run Section 7 for feedback fine-tuning")

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 8B — Autopilot Diagnostics
# ═══════════════════════════════════════════════════════
print("AUTOPILOT DIAGNOSTICS")
print("=" * 60)

# Check 1/5 — GPU
print("\n📊 Check 1/5 — GPU STATUS:")
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated(0) / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {torch.cuda.get_device_properties(0).name}")
    print(f"  Memory: {mem_used:.1f} / {mem_total:.1f} GB")
else:
    print("  ⚠️  No GPU")

# Check 2/5 — Disk
print("\n📊 Check 2/5 — DISK SPACE:")
try:
    stat = os.statvfs('/')
    free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
    total_gb = (stat.f_blocks * stat.f_frsize) / 1e9
    print(f"  Free: {free_gb:.1f} / {total_gb:.1f} GB")
    if free_gb < 5:
        print("  ⚠️  Low disk space!")
except Exception:
    try:
        result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
        print(f"  {result.stdout.strip()}")
    except Exception:
        print("  Could not check disk space")

# Check 3/5 — Training runs
print("\n📊 Check 3/5 — TRAINING RUNS:")
found_runs = False
for search_dir in ["runs/classify", "runs/detect"]:
    runs = sorted(glob.glob(f"{search_dir}/*/"))
    for r in runs:
        has_best = os.path.exists(os.path.join(r, "weights", "best.pt"))
        print(f"  {'✅' if has_best else '⚠️'} {r} {'(best.pt found)' if has_best else '(no best.pt)'}")
        found_runs = True
if not found_runs:
    print("  No training runs found")

# Check 4/5 — Downloaded files
print("\n📊 Check 4/5 — DOWNLOADED FILES:")
pt_files = sorted(glob.glob("*.pt"))
zip_files = sorted(glob.glob("*.zip"))
all_files = pt_files + zip_files
for f in all_files:
    size_mb = os.path.getsize(f) / 1e6
    flag = " ⚠️  suspiciously small!" if size_mb < 1 else ""
    print(f"  {f}: {size_mb:.1f} MB{flag}")
if not all_files:
    print("  No model files found in working directory")

# Check 5/5 — Errors
print("\n📊 Check 5/5 — ERROR SUMMARY:")
errors = [x for x in TRAINING_LOG if x[1] in ('ERROR', 'OOM_FAIL', 'SKIPPED', 'MISSING')]
if errors:
    for name, status, score, detail in errors:
        print(f"  ❌ {name}: {status} — {detail}")
else:
    print("  ✅ No errors in training log")

# Final score
good = len([f for f in all_files if os.path.getsize(f) > 1e6])
total_expected = 6  # 3 VideoMAE zips + 3 YOLO pts
print(f"\n{'=' * 60}")
print(f"SCORE: {good}/{total_expected} models ready")
if good >= total_expected:
    print("🎉 All models trained and downloaded successfully!")
elif good >= total_expected // 2:
    print("⚠️  Some models missing — check errors above")
else:
    print("❌ Most models missing — check GPU, data, and errors above")